PART 02：三道闸门——一道比一道晚，一道比一道贵
整体设计一句话讲完：每个工具调用，执行前依次过三道闸门，全都不命中，才放行执行。


工具调用 → [闸门1: 硬拒绝] → [闸门2: 规则匹配] → [闸门3: 人工审批] → 执行
                ↓命中               ↓命中                ↓拒绝
              直接拦下             转闸门3             直接拦下

              
闸门 1：硬拒绝表——没得商量的

In [ ]:
DENY_LIST = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "dd if=", "> /dev/sda"]

def check_deny_list(command: str) -> str | None:
    for pattern in DENY_LIST:
        if pattern in command:
            return f"Blocked: '{pattern}' is on the deny list"
    return None

一张写死的黑名单，朴素到令人发指的子串匹配：命令文本里包含任何一个词，直接拦，不问用户、不给解释的机会。这些是"任何情况下都不该由一个 Agent 干"的事——格盘、关机、提权、直写磁盘。

闸门 2：规则匹配——"看情况"的操作

In [ ]:
PERMISSION_RULES = [
    # PERMISSION_RULES是一个列表，列表里面每一项是一条安全规则（字典）。可以写多条安全规则。
    {"tools": ["read_file", "write_file", "edit_file"],
    #  当调用的工具是 `read_file` / `write_file` / `edit_file` 这三个文件操作工具时，执行这条校验。
     "check": lambda args: not (WORKDIR / args.get("path", "")).resolve().is_relative_to(WORKDIR),
     "message": "Access outside workspace"},

    {"tools": ["bash"],
    #  触发提示：存在破坏性高危命令
     "check": lambda args: any(kw in args.get("command", "") for kw in ["rm ", "> /etc/", "chmod 777"]),
     "message": "Potentially destructive command"},
]

def check_rules(tool_name: str, args: dict) -> str | None:
    for rule in PERMISSION_RULES:
        if tool_name in rule["tools"] and rule["check"](args):
            return rule["message"]
    return None

每条规则三个字段：管哪些工具、什么条件算命中、命中了说什么。注意条件里的门道——

第一条管文件工具，条件是"路径 resolve 之后落在工作区外"。同一个 read_file，读自己项目里的文件直接过，读 ../../etc/passwd 就命中。权限的粒度是"操作"，不是"工具"：工具没有好坏，操作才有。这在真实产品里是同一套思想：读文件默认放行，写文件默认要问——按危害分级，不按工具分级。

第二条管 bash，盯三个危险词：删文件、改系统配置、离谱放权。命中任何一个，转闸门 3。

闸门 3：人工审批——最贵的一道闸

In [ ]:
def ask_user(tool_name: str, args: dict, reason: str) -> str:
    print(f"\n\033[33m[permission] {reason}\033[0m")
    print(f"   Tool: {tool_name}({args})")
    choice = input("   Allow? [y/N] ").strip().lower()
    return "allow" if choice in ("y", "yes") else "deny"
# 人工二次确认

终端打一行黄字，说明为什么拦你、拦的是哪个调用（连参数原样打出——用户得知道自己在批什么，蒙着眼睛签字的审批等于没有审批），然后停下来等人。

注意 [y/N] 这个写法：大写的 N 是默认值。直接回车，就是拒绝。默认站在安全这一边，想放行必须明确表态。

三道闸门背后，是两条设计哲学
第一句：顺序即成本。 三道闸门从前往后，检查越来越贵：闸门 1 是子串匹配，微秒级；闸门 2 是几条 lambda，可忽略；闸门 3 要打断一个人类——他可能在写代码、在想问题，这一次弹窗的代价顶得上前一亿次字符串匹配。所以最便宜的放最前面，能早早拦下的绝不往后面送。

第二句：默认快，例外慢。 很多人以为权限系统是层层设卡，过五关斩六将才能执行一个命令。反了。看这条管线的结构：三道全不命中，直接执行，零摩擦——而日常操作里，绝大多数调用走的就是这条快车道。读文件、跑测试、改代码，全程不会打扰你一次；只有真正出格的动作，才值得消费一次"人类的时间"。

权限系统不是给所有操作上锁，是给危险操作排队。

![](三道门闸.png)